# Kraken OHLC Data Import

This notebook imports historical OHLC (Open, High, Low, Close) spot data from Kraken file dumps into the PostgreSQL database.

**Features:**
- Parses Kraken CSV files (hourly `*_60.csv` format)
- Detects and skips duplicate data by comparing timestamp ranges
- Bulk inserts using efficient batch operations
- Extracts files from large ZIP archives

**Data Source:** Kraken OHLC file dumps (quarterly or full historical archives)

In [ ]:
from pathlib import Path

from clients.db_client import DBClient
from clients.kraken_csv_client import KrakenCSVClient
import zipfile

kraken = KrakenCSVClient(data_path="../../data/master_q4")

In [ ]:
# Import Kraken CSV data with duplicate detection
csv_files = kraken.load_csv_files(pattern="*_60.csv")
print(f"Found {len(csv_files)} CSV file(s)\n")


with DBClient() as db:
    total_inserted = 0
    total_skipped = 0
    instruments_id_dict = db.get_instruments_by_exchange(exchange="kraken")

    for file_path in csv_files:
        ticker = kraken.get_ticker_from_filename(file_path)
        
        # Check if instrument exists in database
        instrument_id = instruments_id_dict.get(ticker)
        
        if instrument_id:
            # Parse CSV
            df = kraken.parse_csv(file_path)
            csv_start = df['timestamp'].min()
            csv_end = df['timestamp'].max()
            
            # Get existing timestamp range from database
            db_min, db_max = db.get_timestamp_range(instrument_id)
            
            if db_min and db_max:
                print(f"DB range: {db_min} to {db_max}")
                print(f"CSV range: {csv_start} to {csv_end}")
                
                # Filter out timestamps that overlap with existing data
                df = df[(df['timestamp'] < db_min) | (df['timestamp'] > db_max)]
                
                if df.empty:
                    print(f"All data already exists, skipping")
                    total_skipped += 1
                    continue
            else:
                print(f"No existing data, inserting {ticker} {len(df)} rows")
            
            # Add instrument_id and insert
            df['instrument_id'] = instrument_id
            rows = db.insert_market_data(df)
            total_inserted += rows
            print(f"Inserted {rows} rows")

    print(f"Total rows inserted: {total_inserted}")
    print(f"Total files skipped (already imported): {total_skipped}")

In [ ]:
def extract_files_from_zip(zip_path: str, pattern: str, output_dir: str = "../../data"):
    """
    Extract specific files from a large ZIP based on pattern.
    Only extracts matching files, saving memory.
    
    Args:
        zip_path: Path to ZIP file
        pattern: Pattern to match (e.g., 'XBTUSD', '_60.csv')
        output_dir: Where to extract files
    """
    zip_path = Path(zip_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if not zip_path.exists():
        print(f"File not found: {zip_path}")
        return
    
    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            # Find matching files (substring match)
            # Skip macOS metadata files
            matching_files = [
                f for f in zf.namelist() 
                if pattern in f 
                and not f.startswith('__MACOSX')  # Skip macOS folder
                and not f.startswith('._')         # Skip resource forks
            ]
            
            print(f"✓ Found {len(matching_files)} matching file(s)\n")
            
            if not matching_files:
                print("No files match the pattern")
                return
            
            # Extract each matching file
            for filename in matching_files:
                info = zf.getinfo(filename)
                size_mb = info.file_size / (1024**2)
                print(f"Extracting: {filename} ({size_mb:.2f} MB)...")
                
                # Extract to output directory
                zf.extract(filename, output_dir)
                print(f"  ✓ Saved to {output_dir / filename}")
            
            print(f"\n✅ Extracted {len(matching_files)} file(s)")
            
    except zipfile.BadZipFile:
        print("Invalid or corrupted ZIP file")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
# Extract all hourly files (_60.csv) from the ZIP
extract_files_from_zip("../../data/Kraken_OHLCVT.zip", "_60.csv")